# Semester Leaderboard — XGBoost Match Outcome Prediction

This notebook is a tuned XGBoost version of the simplified leaderboard notebook. It uses the current course endpoint, applies the required patch-wise temporal split, builds pre-game-only features, tunes XGBoost on an internal temporal validation split, then reports the Canvas leaderboard table.

Verified local run summary:

- Dataset: `rubiogarciaf/DSCI6523_LeagueofLegends`
- Repository last modified: `2026-07-04 20:51:47+00:00`
- Latest file: `07-03-2026.parquet`
- Data coverage: `2026-05-01` to `2026-07-03`
- Engineered feature count: `98`
- Best validation trial: #3


In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("xgboost") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])

import json
import time

import numpy as np
import pandas as pd
from huggingface_hub import HfApi
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from xgboost import XGBClassifier

In [2]:
REPO_ID = "rubiogarciaf/DSCI6523_LeagueofLegends"
RANDOM_STATE = 42

api = HfApi()
info = api.dataset_info(REPO_ID)
parquet_files = sorted(
    f for f in api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
    if f.endswith(".parquet")
)
base_url = f"https://huggingface.co/datasets/{REPO_ID}/resolve/main"

print(f"Dataset: {REPO_ID}")
print(f"Repository last modified: {info.lastModified}")
print(f"Parquet files: {len(parquet_files)}")
print(f"First file: {parquet_files[0]}")
print(f"Last file: {parquet_files[-1]}")

Dataset: rubiogarciaf/DSCI6523_LeagueofLegends
Repository last modified: 2026-07-04 20:51:47+00:00
Parquet files: 64
First file: 05-01-2026.parquet
Last file: 07-03-2026.parquet


In [3]:
df = pd.concat(
    [pd.read_parquet(f"{base_url}/{file}") for file in parquet_files],
    ignore_index=True,
)
df["gameId"] = df["gameId"].astype(str)
df = df.sort_values(["gameCreation", "gameId"]).reset_index(drop=True)

coverage = pd.DataFrame({
    "field": ["rows", "min gameCreation", "max gameCreation", "min date", "max date", "patches"],
    "value": [
        len(df),
        int(df["gameCreation"].min()),
        int(df["gameCreation"].max()),
        pd.to_datetime(df["gameCreation"].min(), unit="ms", utc=True).strftime("%Y-%m-%d"),
        pd.to_datetime(df["gameCreation"].max(), unit="ms", utc=True).strftime("%Y-%m-%d"),
        ", ".join(str(p) for p in sorted(df["patch"].dropna().unique())),
    ],
})
coverage

,field,value
0,rows,1825332
1,min gameCreation,1777593610000
2,max gameCreation,1783122235000
3,min date,2026-05-01
4,max date,2026-07-03
5,patches,"169, 1610, 1611, 1612, 1613"


## Required leaderboard split

Canvas requires a patch-wise temporal split: within each patch, train on games at or below the 80th percentile of `gameCreation`; test on the rest.

In [4]:
cutoffs = df.groupby("patch")["gameCreation"].quantile(0.8)
df["cutoff"] = df["patch"].map(cutoffs)

train_mask = df["gameCreation"] <= df["cutoff"]
test_mask = ~train_mask
y = df["bwin"].astype("int8")

split_summary = pd.DataFrame({
    "Field": ["Min gameCreation", "Max gameCreation", "Number of games"],
    "Train": [int(df.loc[train_mask, "gameCreation"].min()), int(df.loc[train_mask, "gameCreation"].max()), int(train_mask.sum())],
    "Test": [int(df.loc[test_mask, "gameCreation"].min()), int(df.loc[test_mask, "gameCreation"].max()), int(test_mask.sum())],
})
split_summary

,Field,Train,Test
0,Min gameCreation,1777593610000,1778514014000
1,Max gameCreation,1783031319000,1783122235000
2,Number of games,1460268,365064


## Feature engineering

The current endpoint is a raw pre-game table, so this rebuilds useful pre-game features instead of relying on the older `diff_` snapshot.

Features include:

- team and role-level League Points differences;
- team and role-level mastery differences;
- missing-value counts;
- patch and tier level;
- champion IDs as model features;
- leakage-safe expanding prior win-rate features for champion, champion-position, player, and player-champion histories.

The prior features are calculated in chronological order and exclude the current row, so they do not use the match result being predicted.

In [5]:
def metric_pack(y_true, proba):
    pred = (proba >= 0.5).astype(int)
    return {
        "auc": float(roc_auc_score(y_true, proba)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "brier": float(brier_score_loss(y_true, proba)),
    }


def participant_prior_features(frame, slot):
    is_blue = slot.startswith("b")
    win = frame["bwin"].to_numpy() if is_blue else (1 - frame["bwin"]).to_numpy()
    pos = frame[f"{slot}_teamPosition"].astype(str)
    champ = frame[f"{slot}_championId"].astype(str)
    puuid = frame[f"{slot}_puuid"].astype(str)
    global_rate = float(win.mean())
    out = pd.DataFrame(index=frame.index)

    specs = {
        f"{slot}_champ": champ,
        f"{slot}_champ_pos": champ + "|" + pos,
        f"{slot}_puuid": puuid,
        f"{slot}_puuid_champ": puuid + "|" + champ,
    }
    alphas = {
        f"{slot}_champ": 50.0,
        f"{slot}_champ_pos": 50.0,
        f"{slot}_puuid": 100.0,
        f"{slot}_puuid_champ": 100.0,
    }

    for name, key in specs.items():
        s_win = pd.Series(win, index=frame.index)
        prior_count = key.groupby(key, sort=False).cumcount().astype("float32")
        prior_wins = (s_win.groupby(key, sort=False).cumsum() - s_win).astype("float32")
        alpha = alphas[name]
        prior_wr = (prior_wins + global_rate * alpha) / (prior_count + alpha)
        out[f"{name}_prior_wr"] = prior_wr.astype("float32")
        out[f"{name}_prior_log_count"] = np.log1p(prior_count).astype("float32")
    return out


def add_team_aggregates(features, frame, source_col, short_name):
    blue = frame[[f"b{i}_{source_col}" for i in range(1, 6)]]
    red = frame[[f"r{i}_{source_col}" for i in range(1, 6)]]

    features[f"blue_{short_name}_sum"] = blue.sum(axis=1, skipna=True).astype("float32")
    features[f"red_{short_name}_sum"] = red.sum(axis=1, skipna=True).astype("float32")
    features[f"diff_{short_name}_sum"] = (features[f"blue_{short_name}_sum"] - features[f"red_{short_name}_sum"]).astype("float32")
    features[f"diff_{short_name}_mean"] = (blue.mean(axis=1, skipna=True) - red.mean(axis=1, skipna=True)).astype("float32")
    features[f"diff_{short_name}_median"] = (blue.median(axis=1, skipna=True) - red.median(axis=1, skipna=True)).astype("float32")
    features[f"diff_{short_name}_min"] = (blue.min(axis=1, skipna=True) - red.min(axis=1, skipna=True)).astype("float32")
    features[f"diff_{short_name}_max"] = (blue.max(axis=1, skipna=True) - red.max(axis=1, skipna=True)).astype("float32")
    features[f"blue_{short_name}_missing"] = blue.isna().sum(axis=1).astype("int8")
    features[f"red_{short_name}_missing"] = red.isna().sum(axis=1).astype("int8")

    for i in range(1, 6):
        features[f"role{i}_diff_{short_name}"] = (frame[f"b{i}_{source_col}"] - frame[f"r{i}_{source_col}"]).astype("float32")

In [6]:
def build_features(frame):
    features = pd.DataFrame(index=frame.index)
    features["patch"] = frame["patch"].astype("int16")
    features["average_leaguePoints"] = frame["average_leaguePoints"].astype("float32")

    tier_order = {
        "Iron": 0, "Bronze": 1, "Silver": 2, "Gold": 3, "Platinum": 4,
        "Emerald": 5, "Diamond": 6, "Master": 7, "Grandmaster": 8, "Challenger": 9,
    }
    div_order = {"IV": 0, "III": 1, "II": 2, "I": 3}
    parts = frame["tier"].fillna("").astype(str).str.split(expand=True)
    tier_base = parts[0].map(tier_order).fillna(-1).astype("int8")
    tier_div = parts[1].map(div_order).fillna(0).astype("int8") if parts.shape[1] > 1 else 0
    features["tier_level"] = (tier_base * 4 + tier_div).astype("int8")

    add_team_aggregates(features, frame, "leaguePoints", "lp")
    add_team_aggregates(features, frame, "mpoints", "mastery")

    for i in range(1, 6):
        features[f"role{i}_same_champion"] = (frame[f"b{i}_championId"] == frame[f"r{i}_championId"]).astype("int8")
        features[f"role{i}_blue_champion"] = frame[f"b{i}_championId"].astype("int16")
        features[f"role{i}_red_champion"] = frame[f"r{i}_championId"].astype("int16")

    prior_parts = []
    for slot in [f"b{i}" for i in range(1, 6)] + [f"r{i}" for i in range(1, 6)]:
        prior_parts.append(participant_prior_features(frame, slot))
    prior = pd.concat(prior_parts, axis=1)

    for family in ["champ", "champ_pos", "puuid", "puuid_champ"]:
        blue_wr = [f"b{i}_{family}_prior_wr" for i in range(1, 6)]
        red_wr = [f"r{i}_{family}_prior_wr" for i in range(1, 6)]
        blue_count = [f"b{i}_{family}_prior_log_count" for i in range(1, 6)]
        red_count = [f"r{i}_{family}_prior_log_count" for i in range(1, 6)]

        features[f"diff_{family}_prior_wr_mean"] = (prior[blue_wr].mean(axis=1) - prior[red_wr].mean(axis=1)).astype("float32")
        features[f"diff_{family}_prior_wr_sum"] = (prior[blue_wr].sum(axis=1) - prior[red_wr].sum(axis=1)).astype("float32")
        features[f"diff_{family}_prior_count_mean"] = (prior[blue_count].mean(axis=1) - prior[red_count].mean(axis=1)).astype("float32")

        for i in range(1, 6):
            features[f"role{i}_diff_{family}_prior_wr"] = (prior[f"b{i}_{family}_prior_wr"] - prior[f"r{i}_{family}_prior_wr"]).astype("float32")
            features[f"role{i}_diff_{family}_prior_count"] = (prior[f"b{i}_{family}_prior_log_count"] - prior[f"r{i}_{family}_prior_log_count"]).astype("float32")

    return features.replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")

feature_start = time.time()
X = build_features(df)
print(f"Engineered feature matrix: {X.shape} in {time.time() - feature_start:.1f} seconds")

Engineered feature matrix: (1825332, 98) in 64.9 seconds


## Hyperparameter tuning

Tuning uses an internal temporal validation split inside the required training period. The final Canvas test set is not used for model selection.

In [7]:
train_df = df.loc[train_mask]
inner_cutoffs = train_df.groupby("patch")["gameCreation"].quantile(0.8)
inner_cut = train_df["patch"].map(inner_cutoffs)

fit_mask = train_mask.copy()
valid_mask = train_mask.copy()
fit_mask.loc[train_mask] = train_df["gameCreation"] <= inner_cut
valid_mask.loc[train_mask] = train_df["gameCreation"] > inner_cut

X_fit, y_fit = X.loc[fit_mask], y.loc[fit_mask]
X_valid, y_valid = X.loc[valid_mask], y.loc[valid_mask]
X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

configs = [
    dict(max_depth=3, learning_rate=0.04, n_estimators=700, min_child_weight=20, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.0, reg_lambda=2.0),
    dict(max_depth=4, learning_rate=0.035, n_estimators=800, min_child_weight=30, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=3.0),
    dict(max_depth=5, learning_rate=0.025, n_estimators=900, min_child_weight=40, subsample=0.80, colsample_bytree=0.80, reg_alpha=0.2, reg_lambda=4.0),
    dict(max_depth=3, learning_rate=0.025, n_estimators=1000, min_child_weight=50, subsample=0.90, colsample_bytree=0.90, reg_alpha=0.0, reg_lambda=5.0),
    dict(max_depth=4, learning_rate=0.02, n_estimators=1100, min_child_weight=60, subsample=0.90, colsample_bytree=0.75, reg_alpha=0.5, reg_lambda=5.0),
    dict(max_depth=2, learning_rate=0.05, n_estimators=600, min_child_weight=20, subsample=0.90, colsample_bytree=0.90, reg_alpha=0.0, reg_lambda=2.0),
]

trials = []
best_trial = None
for idx, params in enumerate(configs, 1):
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
    )
    start = time.time()
    model.fit(X_fit, y_fit, verbose=False)
    p_fit = model.predict_proba(X_fit)[:, 1]
    p_valid = model.predict_proba(X_valid)[:, 1]
    trial = {
        "trial": idx,
        "fit_auc": metric_pack(y_fit, p_fit)["auc"],
        "valid_auc": metric_pack(y_valid, p_valid)["auc"],
        "valid_accuracy": metric_pack(y_valid, p_valid)["accuracy"],
        "valid_brier": metric_pack(y_valid, p_valid)["brier"],
        "seconds": time.time() - start,
        "params": params,
    }
    trials.append(trial)
    if best_trial is None or trial["valid_auc"] > best_trial["valid_auc"]:
        best_trial = trial

trials_df = pd.DataFrame(trials).drop(columns=["params"])
print("Best trial:")
print(json.dumps(best_trial, indent=2))
trials_df

Best trial:
{
  "trial": 3,
  "fit_auc": 0.618039773498515,
  "valid_auc": 0.6017942570390085,
  "valid_accuracy": 0.5718913070275157,
  "valid_brier": 0.2418196903152259,
  "seconds": 93.0564911365509,
  "params": {
    "max_depth": 5,
    "learning_rate": 0.025,
    "n_estimators": 900,
    "min_child_weight": 40,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.2,
    "reg_lambda": 4.0
  }
}


,trial,fit_auc,valid_auc,valid_accuracy,valid_brier,seconds
0,1,0.604561,0.599864,0.570333,0.242133,55.628628
1,2,0.611455,0.601323,0.571867,0.241909,73.630617
2,3,0.618040,0.601794,0.571891,0.241820,93.056491
3,4,0.603830,0.599637,0.569772,0.242181,74.507169
4,5,0.608710,0.600748,0.570761,0.241992,97.158856
5,6,0.599252,0.597371,0.568717,0.242568,39.792270


In [8]:
final_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **best_trial["params"],
)
final_model.fit(X_train, y_train, verbose=False)

train_proba = final_model.predict_proba(X_train)[:, 1]
test_proba = final_model.predict_proba(X_test)[:, 1]
train_metrics = metric_pack(y_train, train_proba)
test_metrics = metric_pack(y_test, test_proba)

metrics = pd.DataFrame({
    "Field": ["Min gameCreation", "Max gameCreation", "Number of games", "AUC", "Accuracy", "Brier score"],
    "Train": [
        int(df.loc[train_mask, "gameCreation"].min()),
        int(df.loc[train_mask, "gameCreation"].max()),
        int(train_mask.sum()),
        train_metrics["auc"],
        train_metrics["accuracy"],
        train_metrics["brier"],
    ],
    "Test": [
        int(df.loc[test_mask, "gameCreation"].min()),
        int(df.loc[test_mask, "gameCreation"].max()),
        int(test_mask.sum()),
        test_metrics["auc"],
        test_metrics["accuracy"],
        test_metrics["brier"],
    ],
})

gap_summary = pd.DataFrame({
    "Constraint": ["AUC train - test <= 0.03", "Accuracy train - test <= 0.03", "Brier test - train <= 0.01"],
    "Value": [
        train_metrics["auc"] - test_metrics["auc"],
        train_metrics["accuracy"] - test_metrics["accuracy"],
        test_metrics["brier"] - train_metrics["brier"],
    ],
})
gap_summary["Passes"] = [gap_summary.loc[0, "Value"] <= 0.03, gap_summary.loc[1, "Value"] <= 0.03, gap_summary.loc[2, "Value"] <= 0.01]

count_fields = {"Min gameCreation", "Max gameCreation", "Number of games"}
display_metrics = metrics.copy()
for column in ["Train", "Test"]:
    display_metrics[column] = [
        f"{int(value):,}" if field in count_fields else f"{value:.4f}"
        for field, value in zip(display_metrics["Field"], display_metrics[column])
    ]

display_gaps = gap_summary.copy()
display_gaps["Value"] = display_gaps["Value"].map(lambda value: f"{value:.4f}")

print("Leaderboard metrics")
display(display_metrics)
print("Generalization-gap checks")
display(display_gaps)

Leaderboard metrics


,Field,Train,Test
0,Min gameCreation,"1,777,593,610,000","1,778,514,014,000"
1,Max gameCreation,"1,783,031,319,000","1,783,122,235,000"
2,Number of games,"1,460,268","365,064"
3,AUC,0.6153,0.6052
4,Accuracy,0.5811,0.5745
5,Brier score,0.2397,0.2413


Generalization-gap checks


,Constraint,Value,Passes
0,AUC train - test <= 0.03,0.0101,True
1,Accuracy train - test <= 0.03,0.0066,True
2,Brier test - train <= 0.01,0.0016,True


In [9]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": final_model.feature_importances_,
}).sort_values("importance", ascending=False)

importance.head(25)

,feature,importance
60,diff_champ_pos_prior_wr_sum,0.113757
21,diff_mastery_median,0.098652
47,diff_champ_prior_wr_sum,0.095203
59,diff_champ_pos_prior_wr_mean,0.052904
22,diff_mastery_min,0.042172
87,diff_puuid_champ_prior_count_mean,0.039708
46,diff_champ_prior_wr_mean,0.037790
6,diff_lp_mean,0.022002
5,diff_lp_sum,0.018807
3,blue_lp_sum,0.016431


In [10]:
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
import pandas as pd

# Predict probabilities and class labels
train_proba = final_model.predict_proba(X_train)[:, 1]
test_proba = final_model.predict_proba(X_test)[:, 1]

train_pred = (train_proba >= 0.5).astype(int)
test_pred = (test_proba >= 0.5).astype(int)

# Compute metrics
train_auc = roc_auc_score(y_train, train_proba)
test_auc = roc_auc_score(y_test, test_proba)

train_accuracy = accuracy_score(y_train, train_pred)
test_accuracy = accuracy_score(y_test, test_pred)

train_brier = brier_score_loss(y_train, train_proba)
test_brier = brier_score_loss(y_test, test_proba)

# Leaderboard metrics table
metrics = pd.DataFrame({
    "Field": [
        "Min gameCreation",
        "Max gameCreation",
        "Number of games",
        "AUC",
        "Accuracy",
        "Brier score",
    ],
    "Train": [
        int(df.loc[train_mask, "gameCreation"].min()),
        int(df.loc[train_mask, "gameCreation"].max()),
        int(train_mask.sum()),
        train_auc,
        train_accuracy,
        train_brier,
    ],
    "Test": [
        int(df.loc[test_mask, "gameCreation"].min()),
        int(df.loc[test_mask, "gameCreation"].max()),
        int(test_mask.sum()),
        test_auc,
        test_accuracy,
        test_brier,
    ],
})

# Generalization gap checks required by Canvas
gap_summary = pd.DataFrame({
    "Constraint": [
        "AUC train - test <= 0.03",
        "Accuracy train - test <= 0.03",
        "Brier test - train <= 0.01",
    ],
    "Value": [
        train_auc - test_auc,
        train_accuracy - test_accuracy,
        test_brier - train_brier,
    ],
})

gap_summary["Passes"] = [
    gap_summary.loc[0, "Value"] <= 0.03,
    gap_summary.loc[1, "Value"] <= 0.03,
    gap_summary.loc[2, "Value"] <= 0.01,
]

# Pretty display formatting
count_fields = {"Min gameCreation", "Max gameCreation", "Number of games"}

display_metrics = metrics.copy()
for column in ["Train", "Test"]:
    display_metrics[column] = [
        f"{int(value):,}" if field in count_fields else f"{value:.4f}"
        for field, value in zip(display_metrics["Field"], display_metrics[column])
    ]

display_gaps = gap_summary.copy()
display_gaps["Value"] = display_gaps["Value"].map(lambda value: f"{value:.4f}")

print("Leaderboard metrics")
display(display_metrics)
print("\nGeneralization-gap checks")
display(display_gaps)

Leaderboard metrics


,Field,Train,Test
0,Min gameCreation,"1,777,593,610,000","1,778,514,014,000"
1,Max gameCreation,"1,783,031,319,000","1,783,122,235,000"
2,Number of games,"1,460,268","365,064"
3,AUC,0.6153,0.6052
4,Accuracy,0.5811,0.5745
5,Brier score,0.2397,0.2413



Generalization-gap checks


,Constraint,Value,Passes
0,AUC train - test <= 0.03,0.0101,True
1,Accuracy train - test <= 0.03,0.0066,True
2,Brier test - train <= 0.01,0.0016,True
